# Introduction
In this notebook, we will conduct exploratory data analysis on our given dataset in order to find
patterns with the features available and resolution time. *Note: the dataset used contains all 
incidents **opened** in 2025, including those which went into the next year (for example a case 
opened on New Years Eve)*.

> In a seperate Tableau Public story, we dive into some characteristics of the dataset. The 
> [**NYC311 Resolution Time Analysis**](https://public.tableau.com/app/profile/luis.jaco8183/viz/nyc311_17861374235790/nyc311)
> viz gives an interactive view of datapoints and patterns which can be seen with resolution time.
> Some of the observations made in the viz will be repeated here for brevity.


# Setup

In [21]:
# setup
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [31]:
# grab data
df = pd.read_csv('../data/nyc311_2025.csv', index_col='id')

In [32]:
# # set time columns
df = df.astype({
    'created' : 'datetime64[ns]',
    'closed' : 'datetime64[ns]'
})

In [33]:
df.head()

,created,closed,agency_name,problem,detail,borough,lat,long,method,resolution_time
id,,,,,,,,,,
67351762,2025-12-31 23:59:28,2026-01-01 00:40:32,New York City Police Department,Noise - Residential,Loud Music/Party,MANHATTAN,40.792141,-73.950097,MOBILE,0.684
67344624,2025-12-31 23:59:23,2026-01-01 01:03:42,New York City Police Department,Noise - Residential,Loud Music/Party,MANHATTAN,40.825137,-73.949447,ONLINE,1.072
67346873,2025-12-31 23:59:21,2026-01-01 00:58:13,New York City Police Department,Blocked Driveway,Partial Access,BROOKLYN,40.619995,-73.921167,PHONE,0.981
67353004,2025-12-31 23:59:20,2026-01-01 00:57:31,New York City Police Department,Noise - Residential,Loud Music/Party,BROOKLYN,40.591542,-73.955979,ONLINE,0.970
67350526,2025-12-31 23:59:12,2026-01-01 00:41:16,New York City Police Department,Noise - Commercial,Loud Music/Party,MANHATTAN,40.822593,-73.949525,MOBILE,0.701


In [34]:
df.info()

<class 'pandas.DataFrame'>
Index: 3486292 entries, 67351762 to 63577994
Data columns (total 10 columns):
 #   Column           Dtype         
---  ------           -----         
 0   created          datetime64[ns]
 1   closed           datetime64[ns]
 2   agency_name      str           
 3   problem          str           
 4   detail           str           
 5   borough          str           
 6   lat              float64       
 7   long             float64       
 8   method           str           
 9   resolution_time  float64       
dtypes: datetime64[ns](2), float64(3), str(5)
memory usage: 292.6 MB


# Feature Engineering
There are a couple of features which can be engineered to give details to our problem. Firstly, the
goal of this project is to predict the resolution time given an incident. As such, we should create
this resolution time. We will calculate this in hours.

We can expand our dataset by adding the following columns:

| name  | from      | description  |
|-------|-----------|--------------|
| c_day   | `created` | day created [1-31]   |
| c_hour  | `created` | hour created [0-23]  |
| c_month | `created` | month created [1-12] |
| full_problem | `problem`, `detail` | combined `problem` and `detail` |

The `full_problem` column may seem problematic due to the amount of combations it would create,
but as problems are independent from agency-to-agency, we can implement multiple regressors later.
Each agency will recieve their own regressor, keeping the problem sets self-contained and reducing
sparsity.

In [35]:
# date features
df['c_day'] = df['created'].dt.day
df['c_hour'] = df['created'].dt.hour
df['c_month'] = df['created'].dt.month

In [38]:
# problem + detail
df['full_problem'] = df['problem'].str.cat(df['detail'], sep=': ', na_rep='')

In [37]:
df.describe(include='all')

,created,closed,agency_name,problem,detail,borough,lat,long,method,resolution_time,c_day,c_hour,c_month,full_problem
count,3486292,3486292,3486292,3486292,3409209,3486292,3.486292e+06,3.486292e+06,3486292,3.486292e+06,3.486292e+06,3.486292e+06,3.486292e+06,3486292
unique,NaN,NaN,15,179,874,6,NaN,NaN,5,NaN,NaN,NaN,NaN,998
top,NaN,NaN,New York City Police Department,Illegal Parking,Loud Music/Party,BROOKLYN,NaN,NaN,ONLINE,NaN,NaN,NaN,NaN,Noise - Residential: Loud Music/Party
freq,NaN,NaN,1708737,574070,492683,1030989,NaN,NaN,1513681,NaN,NaN,NaN,NaN,299455
mean,2025-07-05 01:54:27.282831872,2025-07-14 18:15:41.368230912,NaN,NaN,NaN,NaN,4.074098e+01,-7.391993e+01,NaN,2.323539e+02,1.552629e+01,1.310924e+01,6.614375e+00,NaN
min,2025-01-01 00:00:12,2025-01-01 00:05:19,NaN,NaN,NaN,NaN,4.049891e+01,-7.425495e+01,NaN,1.000000e-03,1.000000e+00,0.000000e+00,1.000000e+00,NaN
25%,2025-04-03 09:34:07.750000128,2025-04-11 09:34:01,NaN,NaN,NaN,NaN,4.067548e+01,-7.396327e+01,NaN,1.119000e+00,8.000000e+00,9.000000e+00,4.000000e+00,NaN
50%,2025-07-07 18:33:23,2025-07-17 15:40:32,NaN,NaN,NaN,NaN,4.073336e+01,-7.392248e+01,NaN,7.058000e+00,1.500000e+01,1.300000e+01,7.000000e+00,NaN
75%,2025-10-07 13:56:06.500000,2025-10-15 17:48:58.500000,NaN,NaN,NaN,NaN,4.082310e+01,-7.386854e+01,NaN,6.923300e+01,2.300000e+01,1.800000e+01,1.000000e+01,NaN
max,2025-12-31 23:59:28,2026-07-25 21:02:54,NaN,NaN,NaN,NaN,4.091287e+01,-7.370037e+01,NaN,1.360887e+04,3.100000e+01,2.300000e+01,1.200000e+01,NaN
